In [30]:
import pandas as pd

df = pd.read_csv("data/asset_telemetry.csv")

print(df.shape)

(240, 8)


In [31]:
test_df = df.copy()

In [32]:
from backend.data_pipeline.data_validation import validate_data

validate_data(test_df)

Data validation completed.


True

In [44]:
from backend.data_pipeline.data_preprocessing import preprocess_data

cleaned_df = preprocess_data(test_df)

Data preprocessing completed.
Final dataset shape: (240, 8)


In [45]:
print(cleaned_df.shape)
print(cleaned_df.isnull().sum())
print(cleaned_df.head())

(240, 8)
timestamp_id      0
asset_id          0
hour              0
irradiance_wm2    0
wind_speed_ms     0
temperature_c     0
vibration_mm_s    0
power_mw          0
dtype: int64
   timestamp_id      asset_id  hour  irradiance_wm2  wind_speed_ms  \
0             0  SOLAR_INV_01     0             0.0           6.57   
1             1  SOLAR_INV_01     1             0.0           6.72   
2             2  SOLAR_INV_01     2             0.0           6.26   
3             3  SOLAR_INV_01     3             0.0           7.24   
4             4  SOLAR_INV_01     4             0.0           7.39   

   temperature_c  vibration_mm_s  power_mw  
0          26.26            1.37       0.0  
1          24.89            1.49       0.0  
2          23.85            1.25       0.0  
3          24.45            1.29       0.0  
4          26.45            1.50       0.0  


In [39]:
# from backend.data_pipeline.feature_engineering import (
#     feature_engineering,
#     save_engineered_data
# )

In [46]:
import importlib

import backend.data_pipeline.feature_engineering as fe

importlib.reload(fe)

feature_engineering = fe.feature_engineering
save_engineered_data = fe.save_engineered_data

In [47]:
print("Preprocessed dataset shape:", cleaned_df.shape)

display(cleaned_df.head())

print("\nColumns:")
print(cleaned_df.columns.tolist())

Preprocessed dataset shape: (240, 8)


,timestamp_id,asset_id,hour,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw
0,0,SOLAR_INV_01,0,0.0,6.57,26.26,1.37,0.0
1,1,SOLAR_INV_01,1,0.0,6.72,24.89,1.49,0.0
2,2,SOLAR_INV_01,2,0.0,6.26,23.85,1.25,0.0
3,3,SOLAR_INV_01,3,0.0,7.24,24.45,1.29,0.0
4,4,SOLAR_INV_01,4,0.0,7.39,26.45,1.50,0.0



Columns:
['timestamp_id', 'asset_id', 'hour', 'irradiance_wm2', 'wind_speed_ms', 'temperature_c', 'vibration_mm_s', 'power_mw']


In [48]:
engg_df = feature_engineering(
    cleaned_df,
    window=6,
    min_periods=3
)

print("Feature engineering completed successfully!")

print("\nOriginal shape:", cleaned_df
      .shape)
print("Engineered shape:", engg_df.shape)

Feature engineering completed successfully!

Original shape: (240, 8)
Engineered shape: (240, 31)


In [49]:
engg_df.columns

Index(['timestamp_id', 'asset_id', 'irradiance_wm2', 'wind_speed_ms',
       'temperature_c', 'vibration_mm_s', 'power_mw', 'asset_type', 'hour_sin',
       'hour_cos', 'power_mw_rolling_median', 'power_mw_rolling_mad',
       'power_mw_median_deviation', 'power_mw_modified_zscore',
       'power_mw_zero_mad_deviation', 'vibration_mm_s_rolling_median',
       'vibration_mm_s_rolling_mad', 'vibration_mm_s_median_deviation',
       'vibration_mm_s_modified_zscore', 'vibration_mm_s_zero_mad_deviation',
       'irradiance_wm2_rolling_median', 'irradiance_wm2_rolling_mad',
       'irradiance_wm2_median_deviation', 'irradiance_wm2_modified_zscore',
       'irradiance_wm2_zero_mad_deviation', 'wind_speed_ms_rolling_median',
       'wind_speed_ms_rolling_mad', 'wind_speed_ms_median_deviation',
       'wind_speed_ms_modified_zscore', 'wind_speed_ms_zero_mad_deviation',
       'power_change_mw'],
      dtype='object')

In [51]:
from backend.anomaly_detection.training import train_detector

In [52]:
print("Dataset shape:", engg_df.shape)

display(engg_df.head())

print("\nAsset distribution:")
print(engg_df["asset_id"].value_counts())

print("\nAsset type distribution:")
print(engg_df["asset_type"].value_counts())

Dataset shape: (240, 31)


,timestamp_id,asset_id,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw,asset_type,hour_sin,hour_cos,...,irradiance_wm2_rolling_mad,irradiance_wm2_median_deviation,irradiance_wm2_modified_zscore,irradiance_wm2_zero_mad_deviation,wind_speed_ms_rolling_median,wind_speed_ms_rolling_mad,wind_speed_ms_median_deviation,wind_speed_ms_modified_zscore,wind_speed_ms_zero_mad_deviation,power_change_mw
0,0,SOLAR_INV_01,0.0,6.57,26.26,1.37,0.0,Solar,0.000000,1.000000,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN
1,1,SOLAR_INV_01,0.0,6.72,24.89,1.49,0.0,Solar,0.258819,0.965926,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0.0
2,2,SOLAR_INV_01,0.0,6.26,23.85,1.25,0.0,Solar,0.500000,0.866025,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0.0
3,3,SOLAR_INV_01,0.0,7.24,24.45,1.29,0.0,Solar,0.707107,0.707107,...,NaN,0.0,NaN,0,6.570,NaN,0.670,NaN,0,0.0
4,4,SOLAR_INV_01,0.0,7.39,26.45,1.50,0.0,Solar,0.866025,0.500000,...,NaN,0.0,NaN,0,6.645,NaN,0.745,NaN,0,0.0



Asset distribution:
asset_id
SOLAR_INV_01    120
WTG_07          120
Name: count, dtype: int64

Asset type distribution:
asset_type
Solar    120
Wind     120
Name: count, dtype: int64


In [53]:
# Creating a chronological training and testing datasets

In [54]:
train_parts = []
test_parts = []

for asset_id, group in engg_df.groupby("asset_id"):

    group = group.sort_values("timestamp_id").reset_index(drop=True)

    split_index = int(len(group) * 0.8)

    if split_index < 4 or split_index >= len(group):
        raise ValueError(
            f"Insufficient observations for asset: {asset_id}"
        )

    train_parts.append(group.iloc[:split_index])
    test_parts.append(group.iloc[split_index:])


train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

In [ ]:
# Verifying the split

In [55]:
print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)

print("\nTraining observations per asset:")
display(train_df.groupby("asset_id").size())

print("\nTesting observations per asset:")
display(test_df.groupby("asset_id").size())

Training dataset: (192, 31)
Testing dataset: (48, 31)

Training observations per asset:


asset_id
SOLAR_INV_01    96
WTG_07          96
dtype: int64


Testing observations per asset:


asset_id
SOLAR_INV_01    24
WTG_07          24
dtype: int64

In [56]:
# Training the detector
configuration = train_detector(
    training_df=train_df
)

Detector configuration saved to: F:\DataSets for Artificial Intelligence\greenko assignment\02_renewable_asset_fault_anomaly_copilot\backend\anomaly_detection\model_artifacts\rolling_mad_config.json


In [57]:
# Inspecting the configuration

import json

print(json.dumps(configuration, indent=4))

{
    "method": "Rolling Median/MAD",
    "window_size": 6,
    "min_periods": 3,
    "anomaly_threshold": 3.5,
    "asset_features": {
        "solar": [
            "power_mw",
            "vibration_mm_s",
            "irradiance_wm2"
        ],
        "wind": [
            "power_mw",
            "vibration_mm_s",
            "wind_speed_ms"
        ]
    },
    "fallback_mad": {
        "solar": {
            "power_mw": 0.31699999999999995,
            "vibration_mm_s": 0.07999999999999985,
            "irradiance_wm2": 167.04999999999995
        },
        "wind": {
            "power_mw": 0.2899999999999998,
            "vibration_mm_s": 0.07999999999999985,
            "wind_speed_ms": 0.6000000000000014
        }
    }
}


In [58]:
from pathlib import Path

config_path = (
    Path("backend")
    / "anomaly_detection"
    / "model_artifacts"
    / "rolling_mad_config.json"
)

print("Configuration exists:", config_path.exists())

if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as file:
        saved_configuration = json.load(file)

    print(json.dumps(saved_configuration, indent=4))

Configuration exists: True
{
    "method": "Rolling Median/MAD",
    "window_size": 6,
    "min_periods": 3,
    "anomaly_threshold": 3.5,
    "asset_features": {
        "solar": [
            "power_mw",
            "vibration_mm_s",
            "irradiance_wm2"
        ],
        "wind": [
            "power_mw",
            "vibration_mm_s",
            "wind_speed_ms"
        ]
    },
    "fallback_mad": {
        "solar": {
            "power_mw": 0.31699999999999995,
            "vibration_mm_s": 0.07999999999999985,
            "irradiance_wm2": 167.04999999999995
        },
        "wind": {
            "power_mw": 0.2899999999999998,
            "vibration_mm_s": 0.07999999999999985,
            "wind_speed_ms": 0.6000000000000014
        }
    }
}


In [ ]:
# Verification that original dataset remains unchanged 

In [59]:
print("Engineered dataset shape:", engg_df.shape)
print("Training dataset shape:", train_df.shape)
print("Testing dataset shape:", test_df.shape)

Engineered dataset shape: (240, 31)
Training dataset shape: (192, 31)
Testing dataset shape: (48, 31)


In [60]:
from backend.anomaly_detection.inference import predict_anomalies

In [ ]:
# Anoamly Detection on Test Dataset

In [61]:
predictions_df = predict_anomalies(
    history_df=train_df,
    inference_df=test_df
)

In [62]:
print("Prediction shape:", predictions_df.shape)

display(predictions_df.head())

Prediction shape: (48, 38)


,timestamp_id,asset_id,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw,asset_type,hour_sin,hour_cos,...,wind_speed_ms_modified_zscore,wind_speed_ms_zero_mad_deviation,power_change_mw,power_mw_is_anomaly,vibration_mm_s_is_anomaly,irradiance_wm2_is_anomaly,wind_speed_ms_is_anomaly,anomaly_count,is_anomaly,anomaly_features
0,96,SOLAR_INV_01,0.0,5.24,24.82,1.33,0.0,solar,0.000000,1.000000,...,0.788646,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]
1,97,SOLAR_INV_01,0.0,4.17,23.21,1.45,0.0,solar,0.258819,0.965926,...,-3.584486,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]
2,98,SOLAR_INV_01,0.0,5.70,22.73,1.21,0.0,solar,0.500000,0.866025,...,1.680333,0,0.0,False,True,False,NaN,2,True,"[vibration_mm_s, vibration_mm_s, wind_speed_ms]"
3,99,SOLAR_INV_01,0.0,6.33,22.59,1.36,0.0,solar,0.707107,0.707107,...,2.735472,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]
4,100,SOLAR_INV_01,0.0,6.73,25.76,1.36,0.0,solar,0.866025,0.500000,...,1.752496,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]


In [ ]:
# Checking Detected Anamolies

In [63]:
print("Total observations:", len(predictions_df))

print(
    "\nAnomaly distribution:"
)

print(
    predictions_df["is_anomaly"].value_counts()
)

print("\nAnomalies by asset:")

display(
    predictions_df.groupby("asset_id")["is_anomaly"].sum()
)

Total observations: 48

Anomaly distribution:
is_anomaly
False    43
True      5
Name: count, dtype: int64

Anomalies by asset:


asset_id
SOLAR_INV_01    4
WTG_07          1
Name: is_anomaly, dtype: int64

In [ ]:
# Inspection of observations flagged as anomalous

In [64]:
anomalies_df = predictions_df[
    predictions_df["is_anomaly"]
].copy()

display(anomalies_df)

,timestamp_id,asset_id,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw,asset_type,hour_sin,hour_cos,...,wind_speed_ms_modified_zscore,wind_speed_ms_zero_mad_deviation,power_change_mw,power_mw_is_anomaly,vibration_mm_s_is_anomaly,irradiance_wm2_is_anomaly,wind_speed_ms_is_anomaly,anomaly_count,is_anomaly,anomaly_features
2,98,SOLAR_INV_01,0.0,5.70,22.73,1.21,0.000,solar,0.500000,0.866025,...,1.680333,0,0.000,False,True,False,NaN,2,True,"[vibration_mm_s, vibration_mm_s, wind_speed_ms]"
10,106,SOLAR_INV_01,692.8,7.55,37.61,1.45,1.316,solar,0.500000,-0.866025,...,2.115477,0,0.241,True,False,True,NaN,3,True,"[power_mw, irradiance_wm2, power_mw, wind_spee..."
16,112,SOLAR_INV_01,400.0,8.61,39.76,1.25,0.760,solar,-0.866025,-0.500000,...,1.587059,0,-0.315,True,False,True,NaN,3,True,"[power_mw, irradiance_wm2, power_mw, wind_spee..."
17,113,SOLAR_INV_01,207.1,8.94,37.39,1.41,0.393,solar,-0.965926,-0.258819,...,1.190699,0,-0.367,True,False,True,NaN,3,True,"[power_mw, irradiance_wm2, power_mw, wind_spee..."
25,217,WTG_07,0.0,6.98,23.59,1.45,1.791,wind,0.258819,0.965926,...,-0.738232,0,-0.648,False,True,NaN,False,2,True,"[vibration_mm_s, irradiance_wm2, vibration_mm_s]"


In [66]:
from backend.anomaly_detection.model_validation import (
    evaluate_holdout,
    evaluate_synthetic_anomalies,
    run_model_validation,
)

In [67]:
print("Training observations:", len(train_df))
print("Test observations:", len(test_df))
print("Predictions:", len(predictions_df))

display(predictions_df.head())

Training observations: 192
Test observations: 48
Predictions: 48


,timestamp_id,asset_id,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw,asset_type,hour_sin,hour_cos,...,wind_speed_ms_modified_zscore,wind_speed_ms_zero_mad_deviation,power_change_mw,power_mw_is_anomaly,vibration_mm_s_is_anomaly,irradiance_wm2_is_anomaly,wind_speed_ms_is_anomaly,anomaly_count,is_anomaly,anomaly_features
0,96,SOLAR_INV_01,0.0,5.24,24.82,1.33,0.0,solar,0.000000,1.000000,...,0.788646,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]
1,97,SOLAR_INV_01,0.0,4.17,23.21,1.45,0.0,solar,0.258819,0.965926,...,-3.584486,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]
2,98,SOLAR_INV_01,0.0,5.70,22.73,1.21,0.0,solar,0.500000,0.866025,...,1.680333,0,0.0,False,True,False,NaN,2,True,"[vibration_mm_s, vibration_mm_s, wind_speed_ms]"
3,99,SOLAR_INV_01,0.0,6.33,22.59,1.36,0.0,solar,0.707107,0.707107,...,2.735472,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]
4,100,SOLAR_INV_01,0.0,6.73,25.76,1.36,0.0,solar,0.866025,0.500000,...,1.752496,0,0.0,False,False,False,NaN,0,False,[wind_speed_ms]


In [68]:
holdout_results = evaluate_holdout(predictions_df)

print(json.dumps(holdout_results, indent=4))

{
    "total_observations": 48,
    "detected_anomalies": 5,
    "normal_observations": 43,
    "anomaly_percentage": 10.42,
    "anomalies_by_asset": {
        "SOLAR_INV_01": {
            "observations": 24,
            "anomalies": 4,
            "anomaly_percentage": 16.67
        },
        "WTG_07": {
            "observations": 24,
            "anomalies": 1,
            "anomaly_percentage": 4.17
        }
    },
    "anomalies_by_sensor": {
        "power_mw": 3,
        "vibration_mm_s": 2,
        "irradiance_wm2": 3,
        "wind_speed_ms": 0
    }
}


In [69]:
synthetic_metrics, synthetic_predictions = (
    evaluate_synthetic_anomalies(
        train_df=train_df,
        test_df=test_df,
    )
)

print(json.dumps(synthetic_metrics, indent=4))

{
    "total_observations": 48,
    "true_positives": 2,
    "true_negatives": 41,
    "false_positives": 5,
    "false_negatives": 0,
    "precision": 0.2857,
    "recall": 1.0,
    "f1_score": 0.4444,
    "accuracy": 0.8958,
    "synthetic_anomaly_observations": 2,
    "synthetic_anomaly_detection_rate": 1.0,
    "interpretation": "Metrics are measured against artificially injected anomalies. Unmodified observations are treated as negative examples for this synthetic benchmark; they are not confirmed normal observations."
}


In [70]:
results, synthetic_predictions = run_model_validation(
    train_df=train_df,
    test_df=test_df,
    predictions_df=predictions_df,
)

print(json.dumps(results, indent=4))

Evaluation results saved to: F:\DataSets for Artificial Intelligence\greenko assignment\02_renewable_asset_fault_anomaly_copilot\backend\anomaly_detection\model_artifacts\evaluation
{
    "method": "Rolling Median/MAD",
    "holdout_evaluation": {
        "total_observations": 48,
        "detected_anomalies": 5,
        "normal_observations": 43,
        "anomaly_percentage": 10.42,
        "anomalies_by_asset": {
            "SOLAR_INV_01": {
                "observations": 24,
                "anomalies": 4,
                "anomaly_percentage": 16.67
            },
            "WTG_07": {
                "observations": 24,
                "anomalies": 1,
                "anomaly_percentage": 4.17
            }
        },
        "anomalies_by_sensor": {
            "power_mw": 3,
            "vibration_mm_s": 2,
            "irradiance_wm2": 3,
            "wind_speed_ms": 0
        }
    },
    "synthetic_anomaly_evaluation": {
        "total_observations": 48,
        "true_posi

In [1]:
from backend.app import load_history_data

history_df = load_history_data()

print("Historical data shape:", history_df.shape)
print("\nColumns:")
print(history_df.columns.tolist())

print("\nObservations per asset:")
print(history_df.groupby("asset_id").size())

print("\nLatest historical timestamp per asset:")
print(history_df.groupby("asset_id")["timestamp_id"].max())

NameError: name 'Query' is not defined

In [2]:
from backend.app import load_history_data

history_df = load_history_data()

print(history_df.shape)
print(history_df.groupby("asset_id").size())

ImportError: cannot import name 'load_history_data' from 'backend.app' (f:\DataSets for Artificial Intelligence\greenko assignment\02_renewable_asset_fault_anomaly_copilot\backend\app.py)

In [3]:
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nDoes backend/app.py exist?")
print(Path("backend/app.py").exists())

print("\nDoes the dataset exist?")
print(Path("data/asset_telemetry.csv").exists())

Current working directory:
f:\DataSets for Artificial Intelligence\greenko assignment\02_renewable_asset_fault_anomaly_copilot

Does backend/app.py exist?
True

Does the dataset exist?
True


In [4]:
import backend.app as app

print("Imported app from:")
print(app.__file__)

print("\nFunction exists:")
print(hasattr(app, "load_history_data"))

print("\nAvailable functions:")
print([name for name in dir(app) if "history" in name.lower()])

Imported app from:
f:\DataSets for Artificial Intelligence\greenko assignment\02_renewable_asset_fault_anomaly_copilot\backend\app.py

Function exists:
False

Available functions:
[]


In [5]:
import importlib
import backend.app as app

importlib.reload(app)

print("Function exists:", hasattr(app, "load_history_data"))

Function exists: False


In [7]:
import importlib
import backend.app as app

importlib.reload(app)

print(hasattr(app, "load_history_data"))

True


In [8]:
history_df = app.load_history_data()

print("Historical data shape:", history_df.shape)

print("\nObservations per asset:")
print(history_df.groupby("asset_id").size())

print("\nLatest historical timestamp:")
print(history_df.groupby("asset_id")["timestamp_id"].max())

Historical data shape: (192, 8)

Observations per asset:
asset_id
SOLAR_INV_01    96
WTG_07          96
dtype: int64

Latest historical timestamp:
asset_id
SOLAR_INV_01     95
WTG_07          215
Name: timestamp_id, dtype: int64


In [9]:
print(
    history_df.groupby("asset_id")["timestamp_id"]
    .agg(["min", "max", "count"])
)

              min  max  count
asset_id                     
SOLAR_INV_01    0   95     96
WTG_07        120  215     96
